# Chapter 6 — Tune a physical parameter

TARGET API · CONVERGING · not executable on the current runtime

> **TARGET API / CONVERGING — not executable on the current runtime.**
> This Chapter uses the inline subsystem’s public `ParameterRef`, never
> a private capacitor.

The first three lessons analyzed a fixed 110 fF / 5.8 nH circuit. Here
the physical question changes: which actual capacitor setting best meets
the named resonance target while keeping the same authored circuit
structure?

## Lesson 6.1 — Expose the tunable capacitor

### Rebuild the physical declaration

This recap repeats the ownership-to-boundary sequence before introducing
a numerical request.

In [ ]:
from scnsim import CircuitPlan, ParameterDefinitions, ParameterSpec, components, units as u

inputs = ParameterDefinitions(id="readout_design")
plan = CircuitPlan(id="primitive_resonator")
resonator = plan.subsystem(id="resonator")

To tune this circuit through supported public inputs, the next cell
declares the capacitance and inductance `ParameterRef` handles with
their baselines. Those declarations create public input handles and
baseline values. Their `ParameterSpec` supplies unit and dimensional
validation for the later physical binding, not optimizer bounds. The
following component cell binds each handle to its actual physical part.
Declaring a parameter starts no optimization; the later
`OptimizationVariable` selects only capacitance for this particular
optimization, and that capacitance has a Plan baseline of 110 fF.

In [ ]:
capacitance = inputs.parameter(
    id="capacitance",
    baseline=110.0 * u.fF,
    spec=ParameterSpec(unit=u.fF),
)
inductance = inputs.parameter(
    id="inductance",
    baseline=5.8 * u.nH,
    spec=ParameterSpec(unit=u.nH),
)

The independent definitions are physically adopted only when the
following component cell binds them inside this inline subsystem.
Independent refs may instead bind through physical fields in a root,
inline, or factory declaration; they do not belong to a declaration
scope. An existing Library component publishes an already-bound ref for
lookup through `component.parameter("capacitance")`, without
private-child access. See the [parameter
concept](../../docs/concepts/units-parameters-and-optimization.qmd) for
the broader ownership model.

The registered C and L handles identify public physical inputs, not
electrical branches. The next cell creates the physical capacitor and
inductor branches and binds each one to its respective handle.

In [ ]:
capacitor = resonator.add(
    components.capacitor(id="capacitor", capacitance=capacitance)
)
inductor = resonator.add(
    components.inductor(id="inductor", inductance=inductance)
)

The LC branches share the child terminal bus and ground.

In [ ]:
resonator_bus = resonator.bus(id="terminal")
parallel_lc = resonator.parallel(
    id="parallel_lc",
    start=resonator_bus,
    branches=((capacitor,), (inductor,)),
    end=resonator.ground,
)

In [ ]:
terminal = resonator.expose_pin(id="terminal", at=resonator_bus)

The parent now needs only this terminal, never a private native leaf.

In [ ]:
signal_bus = plan.bus(id="signal_boundary")
resonator_root_bus = plan.bus(id="resonator_node")
coupling_cap = plan.add(
    components.capacitor(id="coupling_cap", capacitance=6.0 * u.fF)
)
coupling = plan.series(
    id="coupling",
    start=signal_bus,
    elements=(coupling_cap,),
    end=resonator_root_bus,
)
plan.link(
    id="resonator_terminal",
    endpoints=(resonator_root_bus, terminal),
)
signal_port = plan.add_port(
    id="signal_in",
    at=signal_bus,
    role="terminated",
    reference_impedance=50.0 * u.ohm,
)
resonator_node = resonator_root_bus.node

`capacitance` and `resonator_node` are the public handles consumed
below.

## Lesson 6.2 — Request and read a tuning search

### Define the optimization request

`ParameterRef` identifies and binds a public physical value; it does not
become the optimizer variable. The request’s `OptimizationVariable`
selects the capacitance ParameterRef with its own bounds. Both
capacitance and inductance ParameterRefs are public, but only
capacitance varies here. The Plan baseline is 110 fF,
`root_hint=6.0 * u.GHz` is a branch locator, the objective target is 6.2
GHz, and the returned best `ParameterSet` is a candidate result.

In [ ]:
from scnsim import (
    CMAESSpec,
    CircuitRun,
    CostObjective,
    DiagonalRootSpec,
    OptimizationSpec,
    OptimizationVariable,
    ParameterSet,
    ReductionPipeline,
)

run = CircuitRun(plan=plan, workspace="workspaces/primitive-course")
quantity_view = run.original.reduce(
    ReductionPipeline().retain(resonator_node)
)
root_spec = DiagonalRootSpec(
    coordinate=resonator_node,
    root_hint=6.0 * u.GHz,
)
optimization_spec = OptimizationSpec(
    variables=(
        OptimizationVariable(
            parameter=capacitance,
            bounds=(80.0 * u.fF, 140.0 * u.fF),
        ),
    ),
    objectives=(
        CostObjective(
            id="resonance_frequency",
            quantity=root_spec.frequency,
            target=6.2 * u.GHz,
            weight=1.0 * u.dimensionless,
        ),
    ),
    optimizer=CMAESSpec(seed=17, max_evaluations=200),
)

In [ ]:
fixed_nonactive = ParameterSet({inductance: 5.8 * u.nH})

In [ ]:
optimization_spec.show()

The displayed specification fixes the objective, bounds, seed, and
evaluation budget before the optimization request is executed.

### Run and read back the winning parameter set

Run the exact request, then keep the winner’s ParameterSet as an
explicit handle for the independent root evaluation.

In [ ]:
from IPython.display import display

optimization = run.optimize(
    quantity_view,
    optimization_spec,
    parameters=fixed_nonactive,
)
best_parameters = optimization.best.parameters
optimization.show()
display(best_parameters)

Use that returned ParameterSet only in the next evaluation; it does not
mutate the Plan baseline.

In [ ]:
winner_root = run.evaluate(
    quantity_view,
    root_spec,
    parameters=best_parameters,
)
display(winner_root.frequency)
winner_root.show()

In [ ]:
from scnsim import CircuitDiagramSpec

winner_diagram = plan.render_schematic(
    CircuitDiagramSpec(),
    parameters=best_parameters,
)
winner_diagram.show()

In [ ]:
winner_diagram.audit.show()

The active optimization variable selects the exposed capacitance
ParameterRef. Candidate binding does not mutate the 110 fF Plan
baseline; `best_parameters` is the returned best ParameterSet used for
this readback.

[Previous](05_sweep_parameters.qmd) · [Next](07_report_resolve.qmd)